# 04 - Construcción del dataset final para modelado

## Objetivo
Definir y construir el conjunto final de datos que será utilizado en el entrenamiento de modelos predictivos de AKI, resolviendo redundancias entre variables, consolidando las características derivadas útiles y preparando la matriz de predictores y la variable objetivo.

### 1. Importar librerías

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

### 2. Cargar el dataset enriquecido

In [2]:
file_path = "../data/processed/aki_dataset_fe.csv"

df = pd.read_csv(file_path)

print("Dataset enriquecido cargado correctamente.")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Dataset enriquecido cargado correctamente.
Filas: 1175
Columnas: 16


### 3. Revisión rápida del dataset

In [8]:
df.head()

,Sexo,EdadAproxIngreso,Creatinina_mg_dL,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,AKI_48h,Diff_Creatininas,Edad_ge_65,N_Farmacos_Riesgo
0,0,61,0.96,4.83,144.0,0.96,0,0,0,0,0,0,0,0.0,0,0
1,0,46,1.15,4.41,142.0,1.15,0,0,0,0,0,0,0,0.0,0,0
2,1,25,0.75,4.09,138.0,0.75,1,0,0,0,0,0,0,0.0,0,1
3,0,78,0.97,4.70,143.0,0.97,0,0,0,0,0,0,0,0.0,1,0
4,1,35,0.63,4.43,140.0,0.63,0,0,0,0,0,0,0,0.0,0,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1175 entries, 0 to 1174
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Sexo                1175 non-null   int64  
 1   EdadAproxIngreso    1175 non-null   int64  
 2   Creatinina_mg_dL    1170 non-null   float64
 3   Potasio_mmol_L      1122 non-null   float64
 4   Sodio_mmol_L        1126 non-null   float64
 5   BaselineCreatinina  1173 non-null   float64
 6   AINE                1175 non-null   int64  
 7   IECA                1175 non-null   int64  
 8   ARAII               1175 non-null   int64  
 9   DIURETICO           1175 non-null   int64  
 10  VANCOMICINA         1175 non-null   int64  
 11  AMINOGLUCOSIDO      1175 non-null   int64  
 12  AKI_48h             1175 non-null   int64  
 13  Diff_Creatininas    1170 non-null   float64
 14  Edad_ge_65          1175 non-null   int64  
 15  N_Farmacos_Riesgo   1175 non-null   int64  
dtypes: float64(5), in

In [5]:
missing_summary = pd.DataFrame({
    "n_missing": df.isnull().sum(),
    "percent_missing": (df.isnull().mean() * 100).round(2)
}).sort_values(by="percent_missing", ascending=False)

missing_summary

,n_missing,percent_missing
Potasio_mmol_L,53,4.51
Sodio_mmol_L,49,4.17
Creatinina_mg_dL,5,0.43
Diff_Creatininas,5,0.43
BaselineCreatinina,2,0.17
Sexo,0,0.00
AINE,0,0.00
EdadAproxIngreso,0,0.00
IECA,0,0.00
ARAII,0,0.00


### 4. Eliminiación de variables redundantes o auxiliares
En este paso se eliminaron del dataset enriquecido aquellas variables que no debían formar parte del conjunto final de modelado.

Se excluyeron específicamente:

- `Creatinina_mg_dL`
- `Diff_Creatininas`

La exclusión de `Creatinina_mg_dL` se decidió en el análisis realizado en el notebook anterior, donde se observó una correlación de 1.00 con `BaselineCreatinina` y una coincidencia exacta en el 99.57% de las observaciones. Esto evidenció una redundancia prácticamente total entre ambas variables, por lo que mantener las dos en el modelo habría supuesto duplicar información sin aportar valor predictivo real.

La variable `Diff_Creatininas`, por su parte, había sido creada únicamente con fines analíticos para cuantificar la discrepancia entre ambas medidas de creatinina. Al no representar un predictor clínicamente interpretable ni una variable disponible en el escenario de predicción, se decidió eliminarla del conjunto final.

In [9]:
cols_to_drop = [
    "Creatinina_mg_dL",
    "Diff_Creatininas"
]

df_model = df.drop(columns=cols_to_drop, errors="ignore").copy()

print("Columnas tras eliminar redundancias/auxiliares:")
print(df_model.columns.tolist())

Columnas tras eliminar redundancias/auxiliares:
['Sexo', 'EdadAproxIngreso', 'Potasio_mmol_L', 'Sodio_mmol_L', 'BaselineCreatinina', 'AINE', 'IECA', 'ARAII', 'DIURETICO', 'VANCOMICINA', 'AMINOGLUCOSIDO', 'AKI_48h', 'Edad_ge_65', 'N_Farmacos_Riesgo']


### 5. Definición del conjunto final de predictores
A continuación, se definió explícitamente el conjunto de variables que integrarían el dataset final para modelado.

Las variables seleccionadas fueron: `Sexo`, `EdadAproxIngreso`, `Edad_ge_65`, `Potasio_mmol_L`, `Sodio_mmol_L`, `BaselineCreatinina`, `AINE`, `IECA`, `ARAII`, `DIURETICO`, `VANCOMICINA`, `AMINOGLUCOSIDO`, `N_Farmacos_Riesgo`

A este conjunto se añadió la variable objetivo: `AKI_48h`

La selección final buscó mantener un equilibrio entre:
- relevancia clínica,
- interpretabilidad,
- simplicidad metodológica,
- y control del riesgo de sobreajuste.

Se conservaron tanto variables clínicas directas como variables derivadas simples (`Edad_ge_65` y `N_Farmacos_Riesgo`), al considerarse potencialmente útiles y fácilmente defendibles desde el punto de vista clínico.

In [10]:
target_col = "AKI_48h"

final_features = [
    "Sexo",
    "EdadAproxIngreso",
    "Edad_ge_65",
    "Potasio_mmol_L",
    "Sodio_mmol_L",
    "BaselineCreatinina",
    "AINE",
    "IECA",
    "ARAII",
    "DIURETICO",
    "VANCOMICINA",
    "AMINOGLUCOSIDO",
    "N_Farmacos_Riesgo"
]

final_cols = final_features + [target_col]

### 6. Creación del dataset final de modelado
Con el listado de variables definido, se construyó un nuevo DataFrame denominado `df_final`, que contiene exclusivamente las variables seleccionadas para modelado y la variable objetivo.


In [11]:
df_final = df_model[final_cols].copy()

print("Dimensiones del dataset final de modelado:")
print(df_final.shape)

df_final.head()

Dimensiones del dataset final de modelado:
(1175, 14)


,Sexo,EdadAproxIngreso,Edad_ge_65,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,N_Farmacos_Riesgo,AKI_48h
0,0,61,0,4.83,144.0,0.96,0,0,0,0,0,0,0,0
1,0,46,0,4.41,142.0,1.15,0,0,0,0,0,0,0,0
2,1,25,0,4.09,138.0,0.75,1,0,0,0,0,0,1,0
3,0,78,1,4.70,143.0,0.97,0,0,0,0,0,0,0,0
4,1,35,0,4.43,140.0,0.63,0,0,0,0,0,0,0,0


#### Resultado
El dataset final quedó formado por:

- **1175 filas**
- **14 columnas**

de las cuales:
- **13 corresponden a predictores**
- **1 corresponde al target (`AKI_48h`)**

### Interpretación
Este paso consolidó el conjunto final de datos que será utilizado en la fase de entrenamiento y evaluación de modelos, manteniendo intacto el número de observaciones de la cohorte y reduciendo el espacio de variables a una estructura compacta y coherente.


### 7. Revisión de estructura y valores faltantes
Una vez construido `df_final`, se revisó su estructura general mediante `info()` y se generó un resumen de valores faltantes.

El dataset final presentó:

- 1175 observaciones
- 14 columnas
- tipos de datos consistentes:
  - variables continuas en `float64`
  - variables binarias y enteras en `int64`

Los valores faltantes residuales se concentraron en tres variables:

- `Potasio_mmol_L`: 53 nulos (4.51%)
- `Sodio_mmol_L`: 49 nulos (4.17%)
- `BaselineCreatinina`: 2 nulos (0.17%)

El resto de las variables no presentaron valores faltantes.

Este patrón de missing se consideró manejable y compatible con estrategias de imputación simples en la fase de modelado. La ausencia de valores faltantes en el target y en la mayoría de predictores binarios refuerza la calidad del dataset final y facilita la construcción de pipelines reproducibles en etapas posteriores.

In [12]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 1175 entries, 0 to 1174
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Sexo                1175 non-null   int64  
 1   EdadAproxIngreso    1175 non-null   int64  
 2   Edad_ge_65          1175 non-null   int64  
 3   Potasio_mmol_L      1122 non-null   float64
 4   Sodio_mmol_L        1126 non-null   float64
 5   BaselineCreatinina  1173 non-null   float64
 6   AINE                1175 non-null   int64  
 7   IECA                1175 non-null   int64  
 8   ARAII               1175 non-null   int64  
 9   DIURETICO           1175 non-null   int64  
 10  VANCOMICINA         1175 non-null   int64  
 11  AMINOGLUCOSIDO      1175 non-null   int64  
 12  N_Farmacos_Riesgo   1175 non-null   int64  
 13  AKI_48h             1175 non-null   int64  
dtypes: float64(3), int64(11)
memory usage: 128.6 KB


In [13]:
final_missing_summary = pd.DataFrame({
    "n_missing": df_final.isnull().sum(),
    "percent_missing": (df_final.isnull().mean() * 100).round(2)
}).sort_values(by="percent_missing", ascending=False)

final_missing_summary

,n_missing,percent_missing
Potasio_mmol_L,53,4.51
Sodio_mmol_L,49,4.17
BaselineCreatinina,2,0.17
Sexo,0,0.00
EdadAproxIngreso,0,0.00
Edad_ge_65,0,0.00
AINE,0,0.00
IECA,0,0.00
ARAII,0,0.00
DIURETICO,0,0.00


### 8. Verificación de distribución del target

Se comprobó nuevamente la distribución de la variable objetivo `AKI_48h` en el dataset final.

El objetivo de esta revisión fue confirmar que:
- la variable objetivo no había sido alterada durante las transformaciones previas,
- el fuerte desbalanceo del problema seguía presente,
- y el dataset final mantenía la misma estructura clínica y estadística observada desde el inicio del proyecto.

Este paso fue importante porque el comportamiento del target condiciona directamente la estrategia de entrenamiento, validación y evaluación de los modelos en la siguiente fase.


In [14]:
print(df_final["AKI_48h"].value_counts(dropna=False))
print(df_final["AKI_48h"].value_counts(normalize=True, dropna=False) * 100)

AKI_48h
0    1146
1      29
Name: count, dtype: int64
AKI_48h
0    97.531915
1     2.468085
Name: proportion, dtype: float64


### 9. Separación en matriz de predictores y variable objetivo
Se realizó la separación formal entre:

- `X`: matriz de predictores
- `y`: variable objetivo

Las dimensiones obtenidas fueron:

- `X`: **(1175, 13)**
- `y`: **(1175,)**

Este paso marcó la transición definitiva desde la preparación del dataset hacia la fase de modelado. La separación explícita de `X` e `y` deja estructurado el problema en el formato estándar requerido por los algoritmos de machine learning supervisado.

Además, confirmó que no se habían perdido filas durante el proceso de creación del dataset final y que el número de predictores coincidía con lo previsto.


In [16]:
X = df_final[final_features].copy()
y = df_final[target_col].copy()

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

Dimensiones de X: (1175, 13)
Dimensiones de y: (1175,)


### 10. Revisión rápida de X e Y
Se inspeccionaron las primeras filas de `X` y `y` para comprobar visualmente que:

- los predictores seleccionados se encontraban correctamente alineados,
- la variable objetivo estaba separada de forma adecuada,
- no existían columnas inesperadas,
- y la estructura final era coherente con la selección establecida en los pasos anteriores.

Esta verificación visual actuó como control final de integridad antes de guardar el dataset. Aunque simple, este paso es útil para detectar errores de selección de columnas o desajustes entre predictores y variable objetivo.


In [19]:
X.head()

,Sexo,EdadAproxIngreso,Edad_ge_65,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,N_Farmacos_Riesgo
0,0,61,0,4.83,144.0,0.96,0,0,0,0,0,0,0
1,0,46,0,4.41,142.0,1.15,0,0,0,0,0,0,0
2,1,25,0,4.09,138.0,0.75,1,0,0,0,0,0,1
3,0,78,1,4.70,143.0,0.97,0,0,0,0,0,0,0
4,1,35,0,4.43,140.0,0.63,0,0,0,0,0,0,0


In [20]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: AKI_48h, dtype: int64

### 11. Guardado del dataset final de modelado
El dataset final fue almacenado en la carpeta `data/processed/` con el nombre:

`aki_dataset_model_ready.csv`

In [21]:
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "aki_dataset_model_ready.csv")

df_final.to_csv(output_path, index=False)

print(f"Dataset final de modelado guardado en: {output_path}")

Dataset final de modelado guardado en: ../data/processed\aki_dataset_model_ready.csv


### 12. Conclusiones del notebook

#### Conclusiones de la construcción del dataset final

En este notebook se definió el conjunto final de variables que será utilizado en el entrenamiento de modelos predictivos de AKI.

Las principales decisiones adoptadas fueron:

- exclusión de `Creatinina_mg_dL` por redundancia extrema con `BaselineCreatinina`,
- exclusión de la variable auxiliar `Diff_Creatininas`,
- conservación de variables derivadas simples y clínicamente interpretables,
- consolidación de un conjunto final de predictores compacto y metodológicamente controlado,
- separación explícita entre matriz de entrada (`X`) y variable objetivo (`y`),
- y guardado del dataset final preparado para modelado.

Como resultado, se obtuvo un dataset final de modelado coherente, interpretable y listo para ser utilizado en la fase de entrenamiento y evaluación de modelos.

En este notebook se completó la construcción del dataset final que se utilizará en la fase de entrenamiento de modelos predictivos de AKI.

Las decisiones más importantes fueron:

- excluir `Creatinina_mg_dL` por redundancia extrema con `BaselineCreatinina`,
- excluir `Diff_Creatininas` por ser una variable auxiliar de análisis,
- consolidar un conjunto final de 13 predictores clínicamente relevantes,
- verificar la estructura y completitud del dataset final,
- separar explícitamente la matriz de predictores y la variable objetivo,
- y guardar una versión final del dataset preparada para modelado.

### Conclusión metodológica principal
Como resultado de este notebook se obtuvo un dataset compacto, interpretable y metodológicamente controlado, sin fugas de información obvias, sin redundancias graves y con un patrón de valores faltantes manejable. Este conjunto final constituye la base directa para el notebook siguiente, dedicado al entrenamiento y evaluación de modelos de clasificación para la predicción temprana de lesión renal aguda.